# NB02 - Captioning (the pairing backbone) - BLIP-2

**Run once, after NB01.** Captions every real image with **BLIP-2** (`Salesforce/blip2-opt-2.7b`), cleans and length-caps each caption deterministically, and stores them in `captions/*.parquet` keyed by `source_real_id`.

This table is what makes the six AI partners share one prompt - every generator reads **these exact captions**, so do not regenerate them per model.

### Why BLIP-2 instead of Florence-2
Florence-2 ships as remote code written for an older `transformers`, which breaks in several ways on Kaggle's current `transformers`. BLIP-2 is a **native** `transformers` model: no `trust_remote_code`, no version pinning, no patches. It loads cleanly on whatever Kaggle provides. Captions are a clean sentence or two (vs Florence-2's longer paragraph); since captions are capped to ~75 CLIP tokens anyway and the goal is content-matched real/AI pairs, this is more than sufficient.

### Prerequisites
- NB00 and NB01 done.
- **Run in a FRESH session** (Run -> Factory reset -> Run All). This avoids the mixed-version package errors from earlier attempts.
- **Internet: ON.**
- **GPU: REQUIRED** (T4 is fine; BLIP-2 in fp16 needs ~7 GB).

Resume-safe: re-running continues from where it stopped. This notebook does NOT upgrade Kaggle's packages -- it uses the preinstalled, mutually-compatible stack as-is.

In [1]:
# ============================================================================
# RUN THIS NOTEBOOK IN A FRESH KAGGLE SESSION:  Run -> Factory reset, then Run All.
# Earlier attempts in a reused session upgraded/downgraded packages and left mixed
# versions (the transformers / Pillow import errors). Those can't be fixed in place --
# only a fresh kernel clears them.
#
# Kaggle's base image already has transformers, torch, torchvision, Pillow, pyarrow,
# huggingface_hub and accelerate, ALL mutually compatible, and that transformers
# supports BLIP-2 natively. So we do NOT upgrade anything (upgrading is what broke
# things); we only install a package if it is genuinely missing, never with -U.
# ============================================================================
import importlib.util, sys, subprocess
required = ["torch", "transformers", "PIL", "pyarrow", "huggingface_hub", "accelerate"]
missing = [m for m in required if importlib.util.find_spec(m) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
    print("installed missing packages:", missing)
else:
    print("all required packages already present -- no installs, no upgrades.")
from importlib.metadata import version
print("transformers", version("transformers"))

all required packages already present -- no installs, no upgrades.
transformers 5.0.0


In [2]:
REPO_ID = "Shanmuk4622/ai-image-detection-dataset"   # <--- SAME as NB00

import os
from huggingface_hub import hf_hub_download
def load_token():
    try:
        from kaggle_secrets import UserSecretsClient
        t = UserSecretsClient().get_secret("HF_TOKEN")
        if t: return t.strip()
    except Exception:
        pass
    for k in ("HF_TOKEN", "HUGGINGFACE_TOKEN", "HUGGING_FACE_HUB_TOKEN"):
        if os.environ.get(k): return os.environ[k].strip()
    import getpass
    return getpass.getpass("HF write token: ").strip()
TOKEN = load_token()

lib = hf_hub_download(REPO_ID, "ai_detect_common.py", repo_type="dataset", token=TOKEN)
import importlib.util
spec = importlib.util.spec_from_file_location("ai_detect_common", lib)
C = importlib.util.module_from_spec(spec); spec.loader.exec_module(C)
cfg = C.read_config(REPO_ID, TOKEN)
print("library + config loaded.")

ai_detect_common.py: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

library + config loaded.


## Load BLIP-2
Native `transformers` classes (`Blip2Processor`, `Blip2ForConditionalGeneration`) - nothing to patch. A CLIP tokenizer is loaded only to count/cap tokens so CLIP-based generators (SD1.5/SDXL) receive the full prompt.

In [3]:
import torch, re
from transformers import Blip2Processor, Blip2ForConditionalGeneration, CLIPTokenizer

device, dtype = C.get_device_dtype()
assert device == "cuda", "Turn the GPU ON for NB02."

CAPTION_MODEL = "Salesforce/blip2-opt-2.7b"   # native; no remote code
proc = Blip2Processor.from_pretrained(CAPTION_MODEL)
try:
    blip = Blip2ForConditionalGeneration.from_pretrained(CAPTION_MODEL, dtype=dtype).to(device).eval()
except TypeError:   # older transformers uses torch_dtype
    blip = Blip2ForConditionalGeneration.from_pretrained(CAPTION_MODEL, torch_dtype=dtype).to(device).eval()
clip_tok = CLIPTokenizer.from_pretrained("openai/clip-vit-large-patch14")
MAXTOK = cfg.get("caption_max_tokens", 75)
CAPTION_TASK = "blip2_caption"
print("BLIP-2 loaded on", device)

@torch.no_grad()
def caption_image(pil):
    # unconditional image captioning: processor returns pixel_values only
    inputs = proc(images=pil, return_tensors="pt").to(device, dtype)
    out = blip.generate(**inputs, max_new_tokens=60, num_beams=5,
                        do_sample=False, repetition_penalty=1.3)
    return proc.batch_decode(out, skip_special_tokens=True)[0].strip()

_LEADINS = ("the image shows ", "the image depicts ", "this image shows ",
            "this image depicts ", "the image is ", "the photo shows ",
            "an image of ", "a photo of ", "the image captures ", "this is ",
            "a picture of ", "there is ", "there are ")
def enhance_caption(raw, max_tokens=MAXTOK):
    s = re.sub(r"\s+", " ", (raw or "").strip())
    low = s.lower()
    for p in _LEADINS:
        if low.startswith(p):
            s = s[len(p):]
            s = (s[:1].upper() + s[1:]) if s else s
            break
    if not s:
        s = "a photograph"
    ids = clip_tok(s, add_special_tokens=False)["input_ids"]
    if len(ids) > max_tokens:
        s = clip_tok.decode(ids[:max_tokens]).strip()
        ids = clip_tok(s, add_special_tokens=False)["input_ids"]
    return s, len(ids)

# quick smoke test on one decoded real image
_probe = C.list_shards(REPO_ID, "real/", TOKEN)
if _probe:
    _loc = C.hf_hub_download(REPO_ID, _probe[0], repo_type="dataset", token=TOKEN)
    _t = C.pq.read_table(_loc, columns=["image"])
    _raw = caption_image(C.decode_png(_t.column("image")[0].as_py()))
    print("sample raw caption:", repr(_raw))
    print("sample enhanced   :", enhance_caption(_raw))

processor_config.json:   0%|          | 0.00/68.0 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/432 [00:00<?, ?B/s]

The image processor of type `BlipImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/882 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

BLIP-2 loaded on cuda


real/real-c5ad8d23-00000.parquet:   0%|          | 0.00/222M [00:00<?, ?B/s]

sample raw caption: 'a lunch box with food in it'
sample enhanced   : ('a lunch box with food in it', 7)


## Caption every real image (resume-safe)
Read the real shards, skip any id already captioned, and append new captions in batched 20-minute commits.

In [4]:
api = C.HfApi(token=TOKEN)
done = C.reconcile_ids(REPO_ID, "captions/", TOKEN)
print("already captioned:", len(done))

cwriter = C.ShardWriter(api, REPO_ID, "captions/", schema=C.CAPTION_SCHEMA,
                        commit_interval=cfg["commit_interval_seconds"],
                        max_rows=cfg["batch_size"], token=TOKEN)
real_shards = C.list_shards(REPO_ID, "real/", TOKEN)
processed = 0
try:
    for f in real_shards:
        local = C.hf_hub_download(REPO_ID, f, repo_type="dataset", token=TOKEN)
        t = C.pq.read_table(local, columns=["image_id", "image"])
        iids = t.column("image_id").to_pylist()
        imgs = t.column("image")
        for k, iid in enumerate(iids):
            if iid in done:
                continue
            pil = C.decode_png(imgs[k].as_py())
            raw = caption_image(pil)
            cap, ntok = enhance_caption(raw)
            cwriter.add(dict(source_real_id=iid, caption=cap, raw_caption=raw,
                             caption_model=CAPTION_MODEL, caption_task=CAPTION_TASK,
                             n_tokens=ntok, created_utc=C.now_utc()))
            done.add(iid); processed += 1
            if processed % 200 == 0:
                print(f"  captioned {processed} this run (total ~{len(done)})")
            cwriter.maybe_flush()
finally:
    cwriter.close()
print("captioning upload complete; this run:", processed)

captions/captions-70411349-00000.parquet:   0%|          | 0.00/23.2k [00:00<?, ?B/s]

already captioned: 500


real/real-c5ad8d23-00001.parquet:   0%|          | 0.00/225M [00:00<?, ?B/s]

  captioned 200 this run (total ~700)
  captioned 400 this run (total ~900)


Processing Files (0 / 0): |          |  0.00B /  0.00B            

...

real/real-c5ad8d23-00099.parquet:   0%|          | 0.00/205M [00:00<?, ?B/s]

  captioned 49200 this run (total ~49700)
  captioned 49400 this run (total ~49900)


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

  committed 500 rows (49500 this run) -> captions/captions-1dccecad-00098.parquet
captioning upload complete; this run: 49500


## Verifier
Confirms every real image has exactly one non-empty caption and none exceed the CLIP limit.

In [5]:
from collections import Counter
real_ids = set()
for f in C.list_shards(REPO_ID, "real/", TOKEN):
    local = C.hf_hub_download(REPO_ID, f, repo_type="dataset", token=TOKEN)
    real_ids.update(C.pq.read_table(local, columns=["image_id"]).column("image_id").to_pylist())

cap_ids, empties, toolong = [], 0, 0
for f in C.list_shards(REPO_ID, "captions/", TOKEN):
    local = C.hf_hub_download(REPO_ID, f, repo_type="dataset", token=TOKEN)
    t = C.pq.read_table(local, columns=["source_real_id", "caption", "n_tokens"])
    cap_ids += t.column("source_real_id").to_pylist()
    empties += sum(1 for c in t.column("caption").to_pylist() if not c or not str(c).strip())
    toolong += sum(1 for n in t.column("n_tokens").to_pylist() if n and n > 77)

dups = [k for k, v in Counter(cap_ids).items() if v > 1]
missing = real_ids - set(cap_ids)
print(f"real={len(real_ids)}  captioned={len(set(cap_ids))}  missing={len(missing)}  "
      f"dups={len(dups)}  empty={empties}  over_77_tokens={toolong}")
print("RESULT:", "PASS" if (not missing and not dups and not empties and not toolong) else "CHECK ABOVE")

captions/captions-1dccecad-00000.parquet:   0%|          | 0.00/23.1k [00:00<?, ?B/s]

captions/captions-1dccecad-00001.parquet:   0%|          | 0.00/23.3k [00:00<?, ?B/s]

captions/captions-1dccecad-00002.parquet:   0%|          | 0.00/23.5k [00:00<?, ?B/s]

captions/captions-1dccecad-00003.parquet:   0%|          | 0.00/23.0k [00:00<?, ?B/s]

captions/captions-1dccecad-00004.parquet:   0%|          | 0.00/23.1k [00:00<?, ?B/s]

...

captions/captions-1dccecad-00095.parquet:   0%|          | 0.00/25.2k [00:00<?, ?B/s]

captions/captions-1dccecad-00096.parquet:   0%|          | 0.00/24.4k [00:00<?, ?B/s]

captions/captions-1dccecad-00097.parquet:   0%|          | 0.00/24.0k [00:00<?, ?B/s]

captions/captions-1dccecad-00098.parquet:   0%|          | 0.00/24.7k [00:00<?, ?B/s]

real=50000  captioned=50000  missing=0  dups=0  empty=0  over_77_tokens=0
RESULT: PASS
